Hesperetin descriptor analysis

In [1]:
from pathlib import Path
import csv, re, subprocess
import numpy as np
from IPython.display import display, Markdown

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "cycle1_opt_freq").is_dir())
analysis = root / "analysis"
def one(folder):
    files = list((root / folder).glob("*.out"))
    if len(files) != 1:
        raise ValueError(f"Expected one final .out in {folder}, found {files}")
    return files[0]
RUNS = {key: one(folder) for key, folder in {
    "opt":"cycle1_opt_freq", "water":"cycle2_fukui/neutral",
    "cation":"cycle2_fukui/cation", "anion":"cycle2_fukui/anion",
    "gas":"cycle2_extension"}.items()}
for key, path in RUNS.items():
    print(key, path.relative_to(root))

opt cycle1_opt_freq\gavino-a_hesperetin_seed232829_cycle1.out
water cycle2_fukui\neutral\gavino-a_hesperetin_seed232829_cycle2.out
cation cycle2_fukui\cation\gavino-a_hesperetin-cation_seed232829_cycle2.out
anion cycle2_fukui\anion\gavino-a_hesperetin-anion_seed232829_cycle2.out
gas cycle2_extension\gavino-a_hesperetin_seed232829_cycle2.out


Read the ORCA outputs

In [2]:
EH_EV = 27.2114
def block_charges(output):
    block = output.split("LOEWDIN ATOMIC CHARGES")[-1].strip().splitlines()
    result = []
    for line in block:
        match = re.match(r"\s*(\d+)\s+([CHO])\s+:\s+(-?[\d.]+)", line)
        if match:
            idx, el, charge = match.groups()
            assert int(idx) == len(result)
            result.append((el, float(charge)))
        elif result:
            break
    assert len(result) == 36
    return result


def coords(output):
    block = output.split("CARTESIAN COORDINATES (ANGSTROEM)")[-1]
    points = []
    for ln in block.splitlines()[2:]:
        if not ln.strip():
            break
        atom, *pos = ln.split()
        points.append([atom, *map(float, pos)])
    assert len(points) == 36
    return points


def neutral_global(output):
    orbital_block = output.split("ORBITAL ENERGIES")[-1]
    orbitals = re.findall(r"^\s*(\d+)\s+([02]\.0000)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s*$", orbital_block, re.M)
    occupied = [r for r in orbitals if r[1] == "2.0000"]
    empty = [r for r in orbitals if r[1] == "0.0000"]
    om, em = occupied[-1], empty[0]
    assert [int(om[0]), int(em[0])] == [78, 79]
    h, l = float(om[2]), float(em[2])
    ip, ea = -h*EH_EV, -l*EH_EV
    gap = ip - ea
    chi, eta = (ip + ea)/2, gap/2
    moment = float(re.findall(r"Magnitude \(Debye\)\s*:\s*([\d.]+)",output)[-1])
    return dict(h_raw=h,l_raw=l,H=-ip,L=-ea,gap=gap,ip=ip,ea=ea,chi=chi,eta=eta,
                omega=chi**2/(2*eta),dn=(7-chi)/(2*eta),dip=moment)

In [3]:
run = {}
for key, name in RUNS.items():
    data = name.read_bytes()
    text = data.decode("utf-16" if data.startswith(b"\xff\xfe") else "utf-8-sig").replace("\r","")
    assert "ORCA TERMINATED NORMALLY" in text and "SCF not fully converged!" not in text
    line = re.findall(r"FINAL SINGLE POINT ENERGY\s+(-?[\d.]+)([^\n]*)",text)[-1]
    assert not line[1].strip()
    rows = block_charges(text)
    rec = dict(name=name,energy=float(line[0]),coords=coords(text),charges=[e[1] for e in rows],
               symbols=[e[0] for e in rows],
               scf=int(re.findall(r"SCF CONVERGED AFTER\s+(\d+) CYCLES",text)[-1]),
               method=re.findall(r"\|\s*\d+>\s*(![^\n]+)",text)[0])
    if key == "water":
        orca_version = re.search(r"Program Version\s+([\d.]+)", text).group(1)
    if key in ("water","gas","opt"):
        rec.update(neutral_global(text))
    if key in ("anion","cation"):
        rec["s2"]=float(re.findall(r"Expectation value of <S\*\*2>\s*:\s*([\d.]+)",text)[-1])
    if key=="opt":
        rec["freq"]=[float(i) for i in re.findall(r"^\s*\d+:\s*(-?\d+\.\d+) cm\*\*-1",text,re.M)]
        assert len(rec["freq"])==108 and min(rec["freq"]) == 0
        assert "THE OPTIMIZATION HAS CONVERGED" in text
    run[key]=rec

assert all(record["coords"]==run["opt"]["coords"] for record in run.values())
symbols = run["water"]["symbols"]
neutral, cation, anion = (np.array(run[k]["charges"]) for k in ("water","cation","anion"))
assert np.allclose([x.sum() for x in (neutral,cation,anion)], [0.,1.,-1.], atol=0.00002)
minus, plus = cation-neutral, neutral-anion
assert np.allclose([minus.sum(),plus.sum()], [1,1], atol=0.00002)

Electronic energies, global descriptors and Fukui indices

In [4]:
def table(headers, rows):
    lines = [" | ".join(headers), " | ".join(["---"] * len(headers))]
    lines += [" | ".join(map(str, row)) for row in rows]
    display(Markdown("\n".join(lines)))

w, g, opt = (run[key] for key in ("water", "gas", "opt"))
table(["Run","Energy (hartree)","Last SCF cycles"], [(k,r["energy"],r["scf"]) for k,r in run.items()])
table(["Descriptor","Water","Gas","Gas − water"], [(k,round(w[k],4),round(g[k],4),round(g[k]-w[k],4)) for k in ("H","L","gap","ip","ea","chi","eta","omega","dn","dip")])
labels = [f"{s}{i}" for i,s in enumerate(symbols)]
table(["Donor","f−","Acceptor","f+"], [(labels[i],round(minus[i],3),labels[j],round(plus[j],3)) for i,j in zip(np.argsort(-minus)[:5],np.argsort(-plus)[:5])])
table(["Atom","Cation","Neutral","Anion","f−","f+"], [(labels[i],cation[i],neutral[i],anion[i],round(minus[i],6),round(plus[i],6)) for i in range(len(labels))])
print("Lowest internal frequency:",min(opt["freq"][6:]),"cm^-1")

Run | Energy (hartree) | Last SCF cycles
--- | --- | ---
opt | -1068.968785547738 | 5
water | -1068.968691342472 | 10
cation | -1068.753993983191 | 13
anion | -1069.041676961642 | 17
gas | -1068.940869100062 | 10

Descriptor | Water | Gas | Gas − water
--- | --- | --- | ---
H | -5.9605 | -6.0709 | -0.1103
L | -1.8106 | -1.718 | 0.0927
gap | 4.1499 | 4.3529 | 0.203
ip | 5.9605 | 6.0709 | 0.1103
ea | 1.8106 | 1.718 | -0.0927
chi | 3.8856 | 3.8944 | 0.0088
eta | 2.075 | 2.1764 | 0.1015
omega | 3.6381 | 3.4843 | -0.1539
dn | 0.7505 | 0.7134 | -0.037
dip | 5.7255 | 4.8007 | -0.9248

Donor | f− | Acceptor | f+
--- | --- | --- | ---
O21 | 0.126 | C10 | 0.193
C6 | 0.122 | O11 | 0.16
O1 | 0.113 | C15 | 0.095
C3 | 0.106 | C13 | 0.075
C2 | 0.096 | C17 | 0.072

Atom | Cation | Neutral | Anion | f− | f+
--- | --- | --- | --- | --- | ---
C0 | -0.057843 | -0.075658 | -0.076599 | 0.017815 | 0.000941
O1 | -0.300999 | -0.413508 | -0.417095 | 0.112509 | 0.003587
C2 | 0.129658 | 0.033768 | 0.026775 | 0.09589 | 0.006993
C3 | 0.169596 | 0.063984 | 0.059486 | 0.105612 | 0.004498
C4 | -0.071512 | -0.12106 | -0.129248 | 0.049548 | 0.008188
C5 | -0.023769 | -0.113506 | -0.113878 | 0.089737 | 0.000372
C6 | -0.008647 | -0.130253 | -0.142508 | 0.121606 | 0.012255
C7 | -0.104383 | -0.150631 | -0.156123 | 0.046248 | 0.005492
C8 | 0.033063 | 0.022507 | 0.01098 | 0.010556 | 0.011527
C9 | -0.180734 | -0.190563 | -0.206849 | 0.009829 | 0.016286
C10 | 0.160694 | 0.156365 | -0.03688 | 0.004329 | 0.193245
O11 | -0.405834 | -0.416357 | -0.576186 | 0.010523 | 0.159829
C12 | -0.156786 | -0.160894 | -0.171733 | 0.004108 | 0.010839
C13 | 0.12965 | 0.126866 | 0.05231 | 0.002784 | 0.074556
C14 | -0.168876 | -0.175782 | -0.209323 | 0.006906 | 0.033541
C15 | 0.112653 | 0.109652 | 0.014939 | 0.003001 | 0.094713
C16 | -0.145537 | -0.152382 | -0.184492 | 0.006845 | 0.03211
C17 | 0.115462 | 0.113635 | 0.041985 | 0.001827 | 0.07165
O18 | -0.335835 | -0.348761 | -0.380046 | 0.012926 | 0.031285
O19 | -0.304811 | -0.308098 | -0.360821 | 0.003287 | 0.052723
O20 | -0.311578 | -0.315852 | -0.37219 | 0.004274 | 0.056338
O21 | -0.198874 | -0.325096 | -0.328873 | 0.126222 | 0.003777
H22 | 0.169693 | 0.150797 | 0.15005 | 0.018896 | 0.000747
H23 | 0.169684 | 0.150861 | 0.150103 | 0.018823 | 0.000758
H24 | 0.165703 | 0.152979 | 0.152247 | 0.012724 | 0.000732
H25 | 0.203651 | 0.182989 | 0.180753 | 0.020662 | 0.002236
H26 | 0.198582 | 0.175384 | 0.173172 | 0.023198 | 0.002212
H27 | 0.195086 | 0.175058 | 0.173144 | 0.020028 | 0.001914
H28 | 0.195911 | 0.186371 | 0.176751 | 0.00954 | 0.00962
H29 | 0.191089 | 0.185892 | 0.163637 | 0.005197 | 0.022255
H30 | 0.200754 | 0.197058 | 0.168649 | 0.003696 | 0.028409
H31 | 0.188086 | 0.186631 | 0.171959 | 0.001455 | 0.014672
H32 | 0.191757 | 0.18996 | 0.175384 | 0.001797 | 0.014576
H33 | 0.281075 | 0.280324 | 0.270398 | 0.000751 | 0.009926
H34 | 0.278718 | 0.277999 | 0.271589 | 0.000719 | 0.00641
H35 | 0.295456 | 0.27932 | 0.278534 | 0.016136 | 0.000786

Lowest internal frequency: 19.45 cm^-1


Conformer table

In [5]:
import json
step = root / "step0_conformers"
def read_csv(name):
    with (step / name).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))
ensemble = read_csv("hesperetin_ensemble.csv")
survivors = read_csv("hesperetin_survivors.csv")
table(list(survivors[0]), [list(row.values()) for row in survivors])
notebook = json.loads((step / "hesperetin_conformer_search.ipynb").read_text(encoding="utf-8"))
log = "".join("".join(o.get("text",[])) for c in notebook["cells"] for o in c.get("outputs",[]))
match = re.search(r"(\d+) distinct minima, (\d+) survivors", log)
if not match:
    raise ValueError("Run and save the conformer notebook with its clustering output visible.")
assert int(match[2]) == len(survivors)
try:
    commit = subprocess.check_output(["git","rev-parse","--short","HEAD"], cwd=root, stderr=subprocess.DEVNULL, text=True).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    commit = "NA"

conformer_label | conformer_id | mmff_energy_kcalmol | relative_energy_kJmol
--- | --- | --- | ---
conf01 | 31 | 34.597106466694484 | 0.0
conf02 | 25 | 35.21862531399012 | 2.600434857084928
conf03 | 30 | 35.96701781617092 | 5.731709086209421
conf04 | 37 | 36.4780190056368 | 7.869738062934662
conf05 | 27 | 36.729193151485894 | 8.920650689167262